# MLPerf-*inspired* Inference — ResNet-50 / ImageNet (Vision) on Colab

Runs the MLPerf `vision/classification_and_detection` reference harness (ResNet-50, PyTorch backend)
under LoadGen on a Colab GPU: **top-1 accuracy + Offline performance**.

> **Not a conformant MLPerf result** — short config + subset data, for quick cross-hardware
> comparison, not submission. See the top-level [README](../../README.md).

Dataset = **Imagenette** (fast.ai's ungated real-ImageNet val subset, 10 classes) since
ImageNet-1k val is access-gated. Expected top-1 ≈ **84–85%** (full 1000-way classifier
restricted to these images).

> Set **Runtime → Change runtime type → GPU** first.

## 1 · GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),
      '|',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2 · Dependencies

torch/torchvision/cv2 are preinstalled on Colab; add loadgen + pycocotools (main.py imports coco).

In [ ]:
!pip -q install "mlcommons-loadgen==6.0.16" "pycocotools==2.0.11" "opencv-python-headless==5.0.0.93"
import mlperf_loadgen, pycocotools, cv2; print('loadgen + pycocotools + cv2 ready')

## 3 · Clone the MLPerf inference repo

In [ ]:
import os, subprocess
INFERENCE_REF='da738a5'   # pin the harness (mlcommons/inference is a moving master)
if not os.path.isdir('/content/inference'):
    subprocess.run(['git','clone','--filter=blob:none','--no-checkout',
                    'https://github.com/mlcommons/inference.git','/content/inference'], check=True)
# Fetch the pin if the cached clone lacks it (best-effort), then hard-reset. ABORT the notebook if the
# reset or the HEAD verification fails — a bare '!git reset' would not stop execution on failure.
subprocess.run(['git','-C','/content/inference','fetch','--filter=blob:none','-q','origin',INFERENCE_REF])
subprocess.run(['git','-C','/content/inference','reset','--hard','-q',INFERENCE_REF], check=True)
head=subprocess.check_output(['git','-C','/content/inference','rev-parse','HEAD']).decode().strip()
want=subprocess.check_output(['git','-C','/content/inference','rev-parse',INFERENCE_REF+'^{commit}']).decode().strip()
assert head==want, f'harness not pinned to {INFERENCE_REF}: HEAD={head}'
print('harness pinned:', head[:12])
%cd /content/inference/vision/classification_and_detection

## 4 · Model + Imagenette dataset

In [ ]:
%%bash
set -euo pipefail
verify(){ echo "$1  $2" | sha256sum -c - >/dev/null || { echo "!! checksum mismatch: $2"; exit 1; }; }
mkdir -p /content/vision && cd /content/vision
[ -s resnet50-19c8e357.pth ] || wget -q -O resnet50-19c8e357.pth 'https://zenodo.org/record/4588417/files/resnet50-19c8e357.pth?download=1'
verify 19c8e3572231adff6824a2da93fd67b5986919a2e65f8b6007eab4edee220097 resnet50-19c8e357.pth
# Imagenette: hash-enforced by DEFAULT against the canonical fast.ai archive (verify BEFORE extract).
# The HASH is the trust anchor, so any mirror matching it is fine; override IMAGENETTE_SHA256 to pin a
# different snapshot. See reference/CHECKSUMS.md.
IMAGENETTE_SHA256="${IMAGENETTE_SHA256:-569b4497c98db6dd29f335d1f109cf315fe127053cedf69010d047f0188e158c}"
if [ ! -d imagenette2-320 ]; then
  wget -q -O i.tgz https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
  verify "$IMAGENETTE_SHA256" i.tgz    # aborts on a bad/partial download or a wrong mirror
  tar xzf i.tgz && rm i.tgz
fi
echo 'val images:' $(find imagenette2-320/val -iname '*.JPEG' | wc -l)

## 5 · Build full model + val_map + patch backend

The Zenodo `.pth` is a legacy-format **state_dict** → build a full torchvision model
(`weights_only=False`). The `pytorch-native` backend returns a bare GPU tensor, which the
post-process can't consume → patch it to return a numpy batch. Build `val_map.txt` with the
10 Imagenette synset → torchvision class-index labels.

In [ ]:
import torch, torchvision, os, glob
# resnet50-19c8e357.pth is SHA-256-verified in the previous cell -> trusted pickle. Legacy tar-format
# state_dict needs weights_only=False; try the safe path first.
PTH='/content/vision/resnet50-19c8e357.pth'
try:
    sd = torch.load(PTH, map_location='cpu', weights_only=True)
except Exception:
    sd = torch.load(PTH, map_location='cpu', weights_only=False)
m = torchvision.models.resnet50(); m.load_state_dict(sd); m.eval()
torch.save(m, '/content/vision/resnet50_full.pth'); print('saved resnet50_full.pth')
# val_map
SYN={'n01440764':0,'n02102040':217,'n02979186':482,'n03000684':491,'n03028079':497,
     'n03394916':566,'n03417042':569,'n03425413':571,'n03445777':574,'n03888257':701}
vd='/content/vision/imagenette2-320/val'; L=[]
for s,i in SYN.items():
    for p in sorted(glob.glob(os.path.join(vd,s,'*.JPEG'))): L.append(f'{os.path.relpath(p,vd)} {i}')
open(os.path.join(vd,'val_map.txt'),'w').write('\n'.join(L)+'\n'); print('val_map entries:',len(L))
# patch backend_pytorch_native to return [numpy batch]
f='python/backend_pytorch_native.py'; s=open(f).read()
old='        with torch.no_grad():\n            output = self.model(feed[key])\n        return output'
new='        with torch.no_grad():\n            output = self.model(feed[key])\n        return [output.cpu().numpy()]'
open(f,'w').write(s.replace(old,new)); print('backend patched:', old.split(chr(10))[-1].strip(),'->','return [output.cpu().numpy()]')

## 6 · Accuracy (top-1)

`main.py` accuracy mode crashes in its post-test percentile stats (numpy2 bug) *after*
LoadGen writes the accuracy log → tolerate it, then score with `tools/accuracy-imagenet.py`.

In [ ]:
%%bash
cat > /content/vision/demo.conf <<'CONF'
resnet50.Offline.target_qps = 700
resnet50.Offline.min_duration = 10000
resnet50.Offline.min_query_count = 1
CONF
cd python
python main.py --profile resnet50-pytorch --backend pytorch-native \
  --model /content/vision/resnet50_full.pth --dataset-path /content/vision/imagenette2-320/val \
  --user_conf /content/vision/demo.conf --scenario Offline --accuracy \
  --output /content/vision/out_acc >/tmp/acc.log 2>&1 || true
echo 'acc log bytes:' $(stat -c%s /content/vision/out_acc/mlperf_log_accuracy.json)
python ../tools/accuracy-imagenet.py --mlperf-accuracy-file /content/vision/out_acc/mlperf_log_accuracy.json \
  --imagenet-val-file /content/vision/imagenette2-320/val/val_map.txt --dtype float32 2>&1 | tail -2

## 7 · Performance (Offline, VALID)

In [ ]:
%%bash
cd python
python main.py --profile resnet50-pytorch --backend pytorch-native \
  --model /content/vision/resnet50_full.pth --dataset-path /content/vision/imagenette2-320/val \
  --user_conf /content/vision/demo.conf --scenario Offline \
  --output /content/vision/out_perf >/tmp/perf.log 2>&1 || true
sed -n '1,12p' /content/vision/out_perf/mlperf_log_summary.txt

## Done ✅ — MLPerf-inspired ResNet-50 vision run on Colab GPU (top-1 + LoadGen-VALID perf under a short config).